# VARUNA AI — Plastic Detection Model Training

Trains a **multi-class** YOLO11 detector on your Roboflow dataset and exports it for the VARUNA app (offline, in-browser ONNX).

**Why multi-class:** the old model merged every class into one "trash" class, so it could never tell plastic from leaves, paper or foam.
This notebook keeps every class. The non-plastic classes teach the model what is *not* plastic, and the app then shows only the plastic ones.

### How to run
1. **Runtime → Change runtime type → T4 GPU** (or better), then Save.
2. **Runtime → Run all.**
3. When asked, allow Google Drive access. Training is saved there, so if Colab disconnects, just **Run all** again and it **resumes** where it stopped.
4. At the end, a zip downloads: `best.onnx` + `varuna_model.json` + charts. Send both files to Claude to plug into the app.

Settings you may want to change are all in **Step 2**.

## Step 1 — GPU check and install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU"
!pip install -q -U ultralytics onnx onnxslim onnxruntime
import ultralytics, torch
ultralytics.checks()
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then Run all again."

## Step 2 — Settings

* **MODEL**: accuracy vs speed in the app.
  `yolo11m` = high accuracy (selected). `yolo11s` = ~2.5× faster in the browser, a little less accurate.
  `yolo11n` = fastest, least accurate.
* **EPOCHS**: upper limit. Training stops early by itself when it stops improving (`PATIENCE`).
* **IMG_SIZE**: 640 matches the app. Don't change unless you know why.

### Dataset — fill in ONE of these (Roboflow blocks direct-link downloads from Colab)
**Option A (recommended):** on the dataset's Roboflow page click **Download Dataset → YOLOv11 → "Show download code"**.
Copy the 4 values from that snippet — `api_key`, `workspace("…")`, `project("…")`, `version(…)` — into `RF_API_KEY`, `RF_WORKSPACE`, `RF_PROJECT`, `RF_VERSION`.
(The `api_key` is YOUR private key — don't share the notebook after pasting it.)

**Option B:** download the dataset zip in your browser (YOLOv11 format), upload it to Google Drive, and set `DRIVE_ZIP` to its path,
e.g. `"/content/drive/MyDrive/plastic_dataset.zip"`.

In [ ]:
# --- Dataset: Option A (Roboflow API) ---
RF_API_KEY   = ""        # from "Show download code": Roboflow(api_key="...")
RF_WORKSPACE = ""        # rf.workspace("...")
RF_PROJECT   = ""        # .project("...")
RF_VERSION   = 1         # .version(N)
# --- Dataset: Option B (zip on Google Drive) ---
DRIVE_ZIP    = ""        # e.g. "/content/drive/MyDrive/plastic_dataset.zip"
# --- Last resort: direct link (usually blocked by Roboflow's bot check on Colab) ---
DATASET_URL  = "https://universe.roboflow.com/ds/Nkk2cI8QAh?key=kkvLaDbHP8"

MODEL      = "yolo11m"   # yolo11n | yolo11s | yolo11m | yolo11l
IMG_SIZE   = 640         # the VARUNA app feeds 640x640
EPOCHS     = 200         # max; early-stops via PATIENCE
PATIENCE   = 40          # stop after this many epochs with no improvement
BATCH      = -1          # -1 = auto-fit to GPU memory
SEED       = 0

USE_DRIVE  = True        # keep checkpoints on Google Drive (survives disconnects)
RUN_NAME   = f"varuna_plastic_{MODEL}_{IMG_SIZE}"

# Any class whose name contains one of these words counts as PLASTIC.
# Step 4 prints the guess — edit PLASTIC_OVERRIDE there if it is wrong.
PLASTIC_KEYWORDS = ["plastic", "bottle", "pet", "bag", "polythene", "polyethylene", "wrapper",
                    "wrap", "packet", "sachet", "cup", "straw", "styrofoam", "foam", "polystyrene",
                    "container", "packaging", "film", "lid", "cap", "tube", "net", "rope", "tarp"]
# Words that mean NOT plastic even if a keyword matched (e.g. "glass bottle").
NOT_PLASTIC_KEYWORDS = ["glass", "metal", "can", "tin", "alumin", "paper", "cardboard", "carton",
                        "wood", "leaf", "organic", "cloth", "fabric", "rubber", "tire", "tyre"]

## Step 3 — Google Drive and dataset download

In [ ]:
import os, glob, shutil, zipfile, subprocess, yaml, random
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/VARUNA_training"
else:
    PROJECT = "/content/runs"
os.makedirs(PROJECT, exist_ok=True)

DS = Path("/content/dataset")
if not (DS.exists() and list(DS.rglob("data.yaml"))):
    shutil.rmtree(DS, ignore_errors=True); DS.mkdir(parents=True)
    if DRIVE_ZIP:                                   # Option B
        assert os.path.exists(DRIVE_ZIP), f"DRIVE_ZIP not found: {DRIVE_ZIP}"
        print("Extracting", DRIVE_ZIP)
        with zipfile.ZipFile(DRIVE_ZIP) as z:
            z.extractall(DS)
    elif RF_API_KEY:                                # Option A
        subprocess.run(["pip", "install", "-q", "roboflow"], check=True)
        from roboflow import Roboflow
        print(f"Downloading {RF_WORKSPACE}/{RF_PROJECT} v{RF_VERSION} via the Roboflow API…")
        rf_project = Roboflow(api_key=RF_API_KEY).workspace(RF_WORKSPACE).project(RF_PROJECT)
        rf_project.version(int(RF_VERSION)).download("yolov11", location=str(DS), overwrite=True)
    else:                                           # last resort
        print("Trying the direct link…")
        zp = "/content/dataset.zip"
        subprocess.run(["curl", "-sL", "-A", "Mozilla/5.0", "-o", zp, DATASET_URL], check=True)
        if not zipfile.is_zipfile(zp):
            os.remove(zp)
            raise SystemExit(
                "\nRoboflow blocked the direct download (bot check). Use Option A or B in Step 2:\n"
                "  A) Roboflow page -> Download Dataset -> YOLOv11 -> 'Show download code', copy api_key /\n"
                "     workspace / project / version into RF_API_KEY, RF_WORKSPACE, RF_PROJECT, RF_VERSION\n"
                "  B) download the zip in your browser, upload to Google Drive, set DRIVE_ZIP\n"
                "Then re-run Step 2 and this cell.")
        with zipfile.ZipFile(zp) as z:
            z.extractall(DS)
        os.remove(zp)
yamls = list(DS.rglob("data.yaml"))
assert yamls, ("No data.yaml in the download. On Roboflow, export the dataset in "
               "'YOLOv11' (or YOLOv8) format and use that download link.")
ROOT = yamls[0].parent
cfg = yaml.safe_load(open(yamls[0]))
names = cfg["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
print("Dataset root:", ROOT)
print(f"{len(names)} classes:", names)

## Step 4 — Dataset check, splits, and which classes are plastic

Counts images and boxes per class, makes a validation split if the dataset has none, and guesses the plastic classes.
**Read the printed plastic list.** If it is wrong, set `PLASTIC_OVERRIDE` to the exact class names and run this cell again.

In [ ]:
PLASTIC_OVERRIDE = None   # e.g. ["plastic bottle", "plastic bag", "wrapper"]

def split_dir(name):
    for cand in (name, "val" if name == "valid" else "valid"):
        p = ROOT / cand / "images"
        if p.exists() and any(p.iterdir()):
            return p
    return None

train_img = split_dir("train"); val_img = split_dir("valid"); test_img = split_dir("test")
assert train_img, "No train/images folder in the dataset."

if val_img is None:   # carve 12% of train out as validation (needed for early stopping)
    print("No validation split — creating one from 12% of train.")
    random.seed(SEED)
    imgs = sorted(p for p in train_img.iterdir() if p.is_file())
    random.shuffle(imgs)
    val_img = ROOT / "valid" / "images"; (ROOT / "valid" / "labels").mkdir(parents=True, exist_ok=True)
    val_img.mkdir(parents=True, exist_ok=True)
    for p in imgs[: max(1, int(len(imgs) * 0.12))]:
        shutil.move(str(p), val_img / p.name)
        lab = train_img.parent / "labels" / (p.stem + ".txt")
        if lab.exists():
            shutil.move(str(lab), ROOT / "valid" / "labels" / lab.name)

def count(img_dir):
    imgs = [p for p in img_dir.iterdir() if p.is_file()] if img_dir else []
    per = [0] * len(names); empty = 0
    for p in imgs:
        lab = img_dir.parent / "labels" / (p.stem + ".txt")
        lines = [l for l in open(lab).read().splitlines() if l.strip()] if lab.exists() else []
        if not lines: empty += 1
        for l in lines:
            c = int(l.split()[0])
            if 0 <= c < len(names): per[c] += 1
    return len(imgs), empty, per

stats = {s: count(d) for s, d in (("train", train_img), ("valid", val_img), ("test", test_img))}
for s, (n, empty, _) in stats.items():
    print(f"{s:6s}: {n:6d} images ({empty} with no objects = background examples)")

import re
def _has(words, keys):   # whole words (plurals ok); long keys also match as prefixes ("alumin" -> "aluminium")
    return any(w == k or w == k + "s" or (len(k) >= 5 and w.startswith(k)) for w in words for k in keys)

def is_plastic(name):
    words = re.findall(r"[a-z]+", name.lower())
    return _has(words, PLASTIC_KEYWORDS) and not _has(words, NOT_PLASTIC_KEYWORDS)

plastic_names = PLASTIC_OVERRIDE or [n for n in names if is_plastic(n)]
unknown = [n for n in plastic_names if n not in names]
assert not unknown, f"PLASTIC_OVERRIDE has names not in the dataset: {unknown}"
PLASTIC_IDS = [names.index(n) for n in plastic_names]

print("\nBoxes per class (train / valid):")
for i, n in enumerate(names):
    tag = "PLASTIC" if i in PLASTIC_IDS else "       "
    warn = "   <-- very few examples, will be weak" if stats["train"][2][i] < 150 else ""
    print(f"  [{tag}] {i:2d} {n:30s} {stats['train'][2][i]:7d} / {stats['valid'][2][i]:6d}{warn}")
print("\nPLASTIC classes:", plastic_names or "NONE — set PLASTIC_OVERRIDE!")
assert PLASTIC_IDS, "No plastic class found. Set PLASTIC_OVERRIDE to the plastic class names and re-run."

# Clean data.yaml with absolute paths (Roboflow's relative paths break in Colab).
DATA_YAML = "/content/varuna_data.yaml"
yaml.safe_dump({"path": str(ROOT), "train": str(train_img), "val": str(val_img),
                **({"test": str(test_img)} if test_img else {}),
                "nc": len(names), "names": names}, open(DATA_YAML, "w"), sort_keys=False)
print("\nWrote", DATA_YAML)

## Step 5 — Train

Settings tuned for **small objects floating on water, seen from a drone**:
vertical flips and rotation (top-down views have no fixed "up"), strong colour/brightness changes (sun glare, murky water, dusk),
scale changes (bottles near and far), mosaic + a little mixup, and a cosine learning-rate schedule.
Starts from COCO-pretrained weights, which already know bottles and cups.

If Colab disconnects: **Run all** again — it resumes from the last saved epoch on Drive.

In [ ]:
from ultralytics import YOLO

run_dir = Path(PROJECT) / RUN_NAME
last = run_dir / "weights" / "last.pt"
best = run_dir / "weights" / "best.pt"

finished = (run_dir / "results.csv").exists() and best.exists() and not last.exists()
if last.exists():
    print("Resuming from", last)
    try:
        YOLO(str(last)).train(resume=True)
    except Exception as e:   # already finished (Ultralytics refuses to resume a completed run)
        print("Resume not needed:", e)
elif finished:
    print("Training already finished:", best)
else:
    model = YOLO(f"{MODEL}.pt")
    model.train(
        data=DATA_YAML, imgsz=IMG_SIZE, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
        project=PROJECT, name=RUN_NAME, exist_ok=True, seed=SEED, deterministic=False,
        optimizer="AdamW", lr0=0.001, cos_lr=True, warmup_epochs=3, weight_decay=0.0005,
        # augmentation for drone views of water
        hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
        degrees=15, translate=0.15, scale=0.6, shear=2.0, perspective=0.0005,
        flipud=0.5, fliplr=0.5, mosaic=1.0, mixup=0.15, close_mosaic=15,
        cache="disk", workers=8, amp=True, plots=True, save_period=-1,
    )
assert best.exists(), "No best.pt found — check the training log above."
print("Best weights:", best)

## Step 6 — How good is it at PLASTIC?

Scores on data the model never trained on (the test split if there is one, otherwise validation).
**mAP50** is the main number: above **0.70** for plastic is good, above **0.80** is excellent.
It also finds the confidence threshold that gives the best plastic precision/recall balance; the app will use it.

In [ ]:
import numpy as np
model = YOLO(str(best))
eval_split = "test" if test_img else "val"
m = model.val(data=DATA_YAML, imgsz=IMG_SIZE, split=eval_split, plots=True,
              project=PROJECT, name=RUN_NAME + "_eval", exist_ok=True)

print(f"\n=== {eval_split.upper()} results ===")
print(f"All classes  mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}")
idx = list(m.box.ap_class_index)
plastic_rows = []
for cid in PLASTIC_IDS:
    if cid in idx:
        j = idx.index(cid)
        p, r, ap50, ap = m.box.class_result(j)
        plastic_rows.append((names[cid], p, r, ap50, ap))
        print(f"  PLASTIC {names[cid]:28s} precision {p:.3f}  recall {r:.3f}  mAP50 {ap50:.3f}")
    else:
        print(f"  PLASTIC {names[cid]:28s} (no examples in {eval_split} split)")
if plastic_rows:
    print(f"\nPLASTIC average mAP50 = {np.mean([x[3] for x in plastic_rows]):.3f}")

# Best confidence threshold for the plastic classes (max mean F1).
REC_CONF = 0.35
try:
    rows = [idx.index(c) for c in PLASTIC_IDS if c in idx]
    f1 = np.asarray(m.box.f1_curve)[rows].mean(0)
    px = np.asarray(m.box.px)
    REC_CONF = float(np.clip(px[int(f1.argmax())], 0.15, 0.7))
    print(f"Recommended app confidence for plastic: {REC_CONF:.2f} (F1 {f1.max():.3f})")
except Exception as e:
    print("Could not compute F1 curve, using", REC_CONF, "-", e)

## Step 7 — See it work (predictions on test images; only plastic boxes drawn)

In [ ]:
from IPython.display import Image as IPImage, display
sample_dir = test_img or val_img
samples = sorted(p for p in sample_dir.iterdir() if p.is_file())[:12]
res = model.predict([str(p) for p in samples], imgsz=IMG_SIZE, conf=REC_CONF, classes=PLASTIC_IDS,
                    save=True, project=PROJECT, name=RUN_NAME + "_samples", exist_ok=True, verbose=False)
for f in sorted((Path(PROJECT) / (RUN_NAME + "_samples")).glob("*.jpg"))[:12]:
    display(IPImage(filename=str(f), width=420))

## Step 8 — Export for the VARUNA app and download

Exports `best.onnx` (640×640, the format the app runs offline) plus `varuna_model.json`, which tells the app the class names,
which ones are plastic, and the confidence to use. Both are also saved to Drive.

In [ ]:
import json, onnxruntime as ort
onnx_path = Path(model.export(format="onnx", imgsz=IMG_SIZE, opset=17, simplify=True, dynamic=False, half=False))

# Sanity-check the ONNX file loads and has the expected output shape [1, 4+nc, N].
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
out = sess.run(None, {sess.get_inputs()[0].name: np.zeros((1, 3, IMG_SIZE, IMG_SIZE), np.float32)})[0]
print("ONNX input:", sess.get_inputs()[0].shape, " output:", out.shape)
assert out.shape[1] == 4 + len(names), "Unexpected output shape"

meta = {
    "model": MODEL, "imgsz": IMG_SIZE, "names": names,
    "plastic_class_ids": PLASTIC_IDS, "plastic_names": plastic_names,
    "recommended_conf": round(REC_CONF, 3),
    "output_layout": "[1, 4+nc, N]: cx,cy,w,h then one score per class; no NMS inside",
    "metrics": {"split": eval_split, "map50_all": round(float(m.box.map50), 4),
                "plastic": {r[0]: {"precision": round(float(r[1]), 4), "recall": round(float(r[2]), 4),
                                   "map50": round(float(r[3]), 4)} for r in plastic_rows}},
}
out_dir = Path("/content/varuna_export"); shutil.rmtree(out_dir, ignore_errors=True); out_dir.mkdir()
shutil.copy(onnx_path, out_dir / "best.onnx")
json.dump(meta, open(out_dir / "varuna_model.json", "w"), indent=2)
for f in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
    if (run_dir / f).exists(): shutil.copy(run_dir / f, out_dir / f)
shutil.copy(best, out_dir / "best.pt")   # keep the PyTorch weights too (for fine-tuning later)

zip_path = shutil.make_archive("/content/varuna_plastic_model", "zip", out_dir)
if USE_DRIVE:
    shutil.copy(zip_path, PROJECT)
    print("Saved to Drive:", Path(PROJECT) / Path(zip_path).name)
print(json.dumps(meta, indent=2))
from google.colab import files
files.download(zip_path)